In [10]:
import pandas as pd
import numpy as np
import os
from matplotlib import rc

rc('mathtext', default='regular')

# === Load data ===
# Streamflow (monthly), one column per station, 'date' column as pandas datetime
root_data = '../../data/'
q_camels  = pd.read_csv(root_data + 'camels/camels_q_mm.csv', parse_dates=['date'])
pr_camels = pd.read_csv(root_data + 'camels/camels_pr_mm.csv', parse_dates=['date'])
et_camels = pd.read_csv(root_data + 'camels/camels_et_mm.csv', parse_dates=['date'])

# === Folder to store results ===
out_dir = os.path.join("../../data/ms_data")
os.makedirs(out_dir, exist_ok=True)

# === Global parameters ===
start = "1960-01-01"
end   = "2025-12-31"
min_obs = 100   # min monthly obs per station to be kept

# === Pre-allocate collectors ===
basin_pr_all = pd.DataFrame()
basin_et_all = pd.DataFrame()
basin_q_all  = pd.DataFrame()

stations = [col for col in q_camels.columns if col != 'date']

# === Build aligned station-by-station tables ===
for cod in stations:
    q_ser = pd.Series(q_camels[cod].values, index=q_camels['date'], name=cod)
    pr_ser = pd.Series(pr_camels[cod].values, index=pr_camels['date'], name=cod)
    et_ser = pd.Series(et_camels[cod].values, index=et_camels['date'],   name=cod)
 
    q_ser  = q_ser.sort_index().loc[start:end]
    pr_ser = pr_ser.sort_index().loc[start:end]
    et_ser = et_ser.sort_index().loc[start:end]

    # Append to dataframes (outer join over time index)
    basin_q_all  = basin_q_all.join(q_ser,  how='outer') if not basin_q_all.empty  else q_ser.to_frame()
    basin_pr_all = basin_pr_all.join(pr_ser, how='outer') if not basin_pr_all.empty else pr_ser.to_frame()
    basin_et_all = basin_et_all.join(et_ser, how='outer') if not basin_et_all.empty else et_ser.to_frame()

# === Compute monthly anomalies (deseasonalize by calendar-month mean) ===
if not basin_q_all.empty:

    q_an  = basin_q_all  - basin_q_all.groupby(basin_q_all.index.month).transform('mean')
    pr_an = basin_pr_all - basin_pr_all.groupby(basin_pr_all.index.month).transform('mean')
    et_an = basin_et_all - basin_et_all.groupby(basin_et_all.index.month).transform('mean')

    # Monthly anomalies
    an_to_save = [
        ("camels_q_an.csv",  q_an),
        ("camels_pr_an.csv", pr_an),
        ("camels_et_an.csv", et_an),
    ]
    for fname, df in an_to_save:
        df = df.sort_index()
        df.index.name = "date"
        df.to_csv(os.path.join(out_dir, fname))

else:
    print("⚠️ No CAMELS basins collected — nothing to save.")

print("Finished!")

Finished!
